In [27]:
from google import genai
from rag_helper import RAGBase
import os
from dotenv import load_dotenv
from sqlitesearch import TextSearchIndex
import json

In [3]:
load_dotenv()

True

In [11]:
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="../faq.db",
)

In [59]:
index.search("i just discovered the llm zoomcamp, can i still join?", num_results=5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': 'fde155ddfb',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I forgot if I registered, can I still join the zoomcamp?',
  'answer': "You don't need to register, as 

In [13]:
client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

In [66]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [71]:
def search(query, num_results=5, index=index):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

available_functions = {
    "search": search
}

In [67]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": (
        "Search the course FAQ database for information relevant to a "
        "student's question. Use concise keyword-based queries. "
        "Call multiple times with different queries when needed."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "Concise keyword-based search query for the course FAQ."
                )
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [68]:
def agent(
    user_input,
    instructions=instructions,
    model="gemini-3.5-flash-lite",
    previous_id=None,
    max_iterations=10
):
    for _ in range(max_iterations):
        interaction = client.interactions.create(
            model=model,
            system_instruction=instructions,
            input=user_input,
            tools=[search_tool],
            previous_interaction_id=previous_id
        )

        function_results = []

        for step in interaction.steps:
            if step.type == "function_call":
                result = available_functions[step.name](**step.arguments)

                print(
                    f"Called {step.name}({step.arguments}) → {result}"
                )

                function_results.append({
                    "type": "function_result",
                    "name": step.name,
                    "call_id": step.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(result)
                        }
                    ],
                })

        if not function_results:
            return interaction.output_text

        user_input = function_results
        previous_id = interaction.id

    raise RuntimeError("Maximum agent iterations exceeded.")

In [72]:
answer = agent("i just discovered the course, can i still join?")

Called search({'query': 'join course late register registration deadline'}) → [{'id': 'cdc3b285e5', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Can I submit homework after the deadline, or get a deadline extension?', 'answer': "No. We don't give individual deadline extensions, and once the homework submission form is closed you can no longer submit it — there are no late submissions. While the form is still open you can submit, even if the listed deadline has already passed.\n\nMissing a homework won't affect your certificate: homework isn't mandatory, only passing the Capstone project is. Homework points only count toward your leaderboard rank, so you'll still appear on the leaderboard with your other submissions."}, {'id': 'a9353fadfe', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?', 'answer': "Yes. As l

In [73]:
print(answer)

Yes, you can still join! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

Are there other areas that you want to explore?
